# 06 · API FastAPI: contrato y comprobaciones

**Propósito.** Documentar el contrato de `src/api/` y verificar que la API carga el pipeline `champion` una sola vez, valida la entrada y responde con una predicción interpretada y la versión del modelo.

| Endpoint | Método | Descripción |
|---|---|---|
| `/health` | GET | Estado, origen del modelo (`mlflow-registry` o `local`), versión y familia |
| `/predict` | POST | Recibe 14 lags y devuelve la predicción del día *t* |
| `/reload` | POST | Recarga el modelo tras una nueva ejecución del flow |
| `/docs` | GET | Swagger UI generado por FastAPI |

**Carga del modelo:** en el `lifespan` de la aplicación (al arrancar) se intenta `models:/{UC_MODEL_NAME}@{MODEL_ALIAS}` en el Registry configurado; si falla, se usa `artifacts/champion_model.pkl`. **No se entrena** al atender solicitudes.

In [1]:
import os, sys, json
from pathlib import Path

# Los notebooks se ejecutan desde la raíz del repositorio para compartir rutas con src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Contrato de entrada y salida (`src/api/schemas.py`)

In [2]:
import inspect
from src.api import schemas
print(inspect.getsource(schemas))

import math
from pydantic import BaseModel, Field, field_validator


class PredictionRequest(BaseModel):
    lags_14_dias: list[float] = Field(
        ...,
        min_length=14,
        max_length=14,
        description="Lista ordenada de vistas diarias de los últimos 14 días (día t-14 al día t-1).",
        examples=[[
            3529320.0, 3816846.0, 1660823.0, 3210395.0, 3791115.0,
            2218409.0, 1034500.0, 3077739.0, 3434652.0, 838785.0,
            1615544.0, 1548594.0, 2100450.0, 1950300.0
        ]],
    )

    @field_validator("lags_14_dias")
    @classmethod
    def validar_vistas_no_negativas(cls, v: list[float]) -> list[float]:
        for idx, val in enumerate(v):
            if math.isnan(val) or math.isinf(val):
                raise ValueError(f"El valor en la posición {idx} no es un número válido.")
            if val < 0:
                raise ValueError(f"Las vistas no pueden ser negativas (valor recibido: {val} en posición {idx}).")
        return v


cla

Reglas de validación de `PredictionRequest.lags_14_dias`:
1. Exactamente 14 valores (`min_length = max_length = 14`), ordenados de `t-14` a `t-1`.
2. Cada valor es numérico, finito y **no negativo**.

Si una regla no se cumple, FastAPI responde **422** con el detalle del error y **no** devuelve una predicción.

## 2. Pruebas en proceso con `TestClient`

In [3]:
from fastapi.testclient import TestClient
from src.api.main import app

cliente = TestClient(app)
cliente.__enter__()  # ejecuta el lifespan: carga el modelo una sola vez
print(json.dumps(cliente.get("/health").json(), indent=2, ensure_ascii=False))

{
  "status": "ok",
  "model_loaded": true,
  "model_source": "mlflow-registry (models:/workspace.default.pcd_otono_2026_tubecast@champion)",
  "model_version": "2",
  "model_family": "random-forest",
  "detail": null
}


In [4]:
valido = {"lags_14_dias": [3529320.0, 3816846.0, 1660823.0, 3210395.0, 3791115.0, 2218409.0, 1034500.0,
                          3077739.0, 3434652.0, 838785.0, 1615544.0, 1548594.0, 2100450.0, 1950300.0]}
resp = cliente.post("/predict", json=valido)
print("HTTP", resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

HTTP 200
{
  "predicted_views": 4599232.566166694,
  "predicted_views_formatted": "4,599,233",
  "interpretation": "Se estima que los videos publicados el día t acumularán 4,599,233 vistas en total, un 90.3% por encima del promedio de los últimos 14 días (2,416,248).",
  "model_name": "workspace.default.pcd_otono_2026_tubecast",
  "model_version": "2",
  "model_family": "random-forest",
  "model_source": "mlflow-registry (models:/workspace.default.pcd_otono_2026_tubecast@champion)"
}


In [5]:
invalidos = {
    "solo 3 valores": {"lags_14_dias": [1.0, 2.0, 3.0]},
    "valor negativo": {"lags_14_dias": [100.0] * 13 + [-5.0]},
    "texto en lugar de número": {"lags_14_dias": ["mucho"] + [100.0] * 13},
    "campo faltante": {"vistas": [100.0] * 14},
}
filas = []
for caso, cuerpo in invalidos.items():
    r = cliente.post("/predict", json=cuerpo)
    filas.append({"caso": caso, "HTTP": r.status_code,
                  "mensaje": r.json()["detail"][0]["msg"],
                  "devuelve_prediccion": "predicted_views" in r.json()})
pd.DataFrame(filas)

,caso,HTTP,mensaje,devuelve_prediccion
0,solo 3 valores,422,"List should have at least 14 items after validation, not 3",False
1,valor negativo,422,"Value error, Las vistas no pueden ser negativas (valor recibido: -5.0 en posición 13).",False
2,texto en lugar de número,422,"Input should be a valid number, unable to parse string as a number",False
3,campo faltante,422,Field required,False


In [6]:
# La predicción de la API coincide con el pipeline del Registry aplicado al mismo DataFrame
import mlflow
from src.api.main import estado, FEATURE_NAMES
X = pd.DataFrame([valido["lags_14_dias"]], columns=FEATURE_NAMES)
assert np.isclose(estado["modelo"].predict(X)[0], resp.json()["predicted_views"])
assert resp.json()["model_version"] == cliente.get("/health").json()["model_version"]
print("Predicción consistente con el pipeline cargado y versión identificada:", resp.json()["model_version"])
cliente.__exit__(None, None, None)

Predicción consistente con el pipeline cargado y versión identificada: 2


**Interpretación de la respuesta.** `predicted_views` es la estimación de las vistas que acumularán los videos publicados el día *t*. `interpretation` la compara con el promedio de la ventana de entrada, y `model_name`/`model_version`/`model_family`/`model_source` identifican exactamente qué versión del Registry respondió. Con entradas inválidas la API responde 422 con un mensaje claro y nunca inventa un valor.

## 3. Proceso externo (uvicorn) y pruebas con curl

Arranque, desde la raíz del repositorio (lee `.env`):

```bash
uv run uvicorn src.api.main:app --host 127.0.0.1 --port 8000
```

Solicitud válida:

```bash
curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d '{"lags_14_dias":[3529320,3816846,1660823,3210395,3791115,2218409,1034500,3077739,3434652,838785,1615544,1548594,2100450,1950300]}'
```

Solicitud inválida (respuesta 422):

```bash
curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d '{"lags_14_dias":[1,2,3]}'
```

La celda siguiente prueba el proceso real si está en ejecución (no lo inicia).

In [7]:
import requests
API_URL = os.getenv("API_URL", "http://127.0.0.1:8000")
try:
    print("GET /health ->", requests.get(f"{API_URL}/health", timeout=5).json())
    r = requests.post(f"{API_URL}/predict", json=valido, timeout=10)
    print("POST /predict (válido) ->", r.status_code, r.json()["predicted_views_formatted"], "v" + r.json()["model_version"])
    r = requests.post(f"{API_URL}/predict", json={"lags_14_dias": [1, 2, 3]}, timeout=10)
    print("POST /predict (inválido) ->", r.status_code, r.json()["detail"][0]["msg"])
except requests.exceptions.ConnectionError:
    print(f"La API no está en ejecución en {API_URL}; arráncala con uvicorn para esta comprobación.")

GET /health -> {'status': 'ok', 'model_loaded': True, 'model_source': 'mlflow-registry (models:/workspace.default.pcd_otono_2026_tubecast@champion)', 'model_version': '2', 'model_family': 'random-forest', 'detail': None}


POST /predict (válido) -> 200 4,599,233 v2
POST /predict (inválido) -> 422 List should have at least 14 items after validation, not 3


**Limitaciones.** La API exige que el cliente provea los 14 lags; no consulta la YouTube API por sí misma. No incluye autenticación (fuera del alcance de esta entrega).

Siguiente: [07-frontend](07-frontend.ipynb).